# ProximityPrep — exploration

Notebook d'exploration de l'étape 3 : distances plage et comptages commerces de proximité.

Objectif : visualiser entrées, enrichissement POI/nearest et remplir `../Output/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "ProximityPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
ROD_OUTPUT = PREPARE / "RodPrep" / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Entrée — hôtels depuis RodPrep

In [ ]:
from proximity_prep.prep import ProximityPrep

prep = ProximityPrep(INPUT_DIR, OUTPUT_DIR)

if not (INPUT_DIR / "hotels.parquet").exists():
    if not (ROD_OUTPUT / "hotel_lookup.parquet").exists():
        raise FileNotFoundError("Exécuter d'abord RodPrep/Explore/explore.ipynb")
    hotels_path = prep.fill_input_from_rod(ROD_OUTPUT)
    print("Entrée créée depuis RodPrep :", hotels_path)
else:
    print("Entrée existante :", INPUT_DIR / "hotels.parquet")

hotels = prep.load_input()
print(f"Hôtels : {len(hotels)}")
hotels[["hotel_code", "hotel_name", "hotel_city", "hotel_lat", "hotel_lon"]].head(10)

## 2. Enrichissement — POI et distances brutes

In [ ]:
enrich_summary = []
for _, hotel in hotels.iterrows():
    code = str(hotel.get("hotel_code", ""))
    name = str(hotel.get("hotel_name", code))
    city = str(hotel.get("hotel_city", ""))
    result = prep._enrich.enrich(hotel_name=name, city=city, hotel_id=code)
    poi = result.features.poi or {}
    nearest = result.features.nearest or {}
    enrich_summary.append({
        "hotel_code": code,
        "source": result.source,
        "poi_fb_100m": poi.get("d_poi_fb_0_0_1km", 0),
        "poi_fb_500m": poi.get("d_poi_fb_0_0_5km", 0),
        "plage_km": nearest.get("d_nearest_beach_km") or nearest.get("nearest_beach_km"),
        "nb_cles_nearest": len(nearest),
    })

enrich_df = pd.DataFrame(enrich_summary)
enrich_df

## 3. Détail POI — premier hôtel

In [ ]:
sample_hotel = hotels.iloc[0]
code = str(sample_hotel.get("hotel_code", ""))
name = str(sample_hotel.get("hotel_name", code))
city = str(sample_hotel.get("hotel_city", ""))

sample_result = prep._enrich.enrich(hotel_name=name, city=city, hotel_id=code)
poi_raw = sample_result.features.poi or {}
nearest_raw = sample_result.features.nearest or {}

print("POI (comptages par rayon)")
pd.DataFrame({"cle": list(poi_raw.keys()), "valeur": list(poi_raw.values())})

In [ ]:
print("Nearest (distances en mètres)")
nearest_items = sorted(nearest_raw.items())
pd.DataFrame({"cle": [k for k, _ in nearest_items[:15]], "valeur": [v for _, v in nearest_items[:15]]})

## 4. Construction ligne par hôtel (logique `run()`)

Renommage en colonnes lisibles : `plage_distance_km`, `commerce_fb_100m`, `distance_{type}_m`…

In [ ]:
built_rows = []
for _, hotel in hotels.iterrows():
    hcode = str(hotel.get("hotel_code", ""))
    hname = str(hotel.get("hotel_name", hcode))
    city = str(hotel.get("hotel_city", ""))
    result = prep._enrich.enrich(hotel_name=hname, city=city, hotel_id=hcode)
    poi = result.features.poi or {}
    nearest = result.features.nearest or {}
    row = {
        "hotel_code": hcode,
        "hotel_name": hname,
        "plage_distance_km": nearest.get("d_nearest_beach_km") or nearest.get("nearest_beach_km"),
        "commerce_fb_100m": poi.get("d_poi_fb_0_0_1km", 0),
        "commerce_fb_500m": poi.get("d_poi_fb_0_0_5km", 0),
        "commerce_non_fb_100m": poi.get("d_poi_not_fb_0_0_1km", 0),
        "commerce_non_fb_500m": poi.get("d_poi_not_fb_0_0_5km", 0),
    }
    for key, value in nearest.items():
        if key.startswith("d_nearest_") and key.endswith("_m"):
            clean = key.replace("d_nearest_", "distance_").replace("_m", "_m")
            row[clean] = value
    built_rows.append(row)

proximity_preview = pd.DataFrame(built_rows)
print(f"Table proximité : {proximity_preview.shape}")
proximity_preview.head()

## 5. Persistance `Output/`

In [ ]:
proximity_final = prep.run()
print(f"proximity : {proximity_final.shape}")
proximity_final.head()

print("\nFichiers produits :")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" ", path.name)